In [25]:
import os
from dotenv import load_dotenv
load_dotenv()

if os.environ["OPENAI_API_KEY"]:
    print("OPENAI_API_KEY is set.")

OPENAI_API_KEY is set.


In [26]:
from langchain_openai import ChatOpenAI



In [27]:
llm = ChatOpenAI(
    model_name="gpt-5-nano",
    temperature=0)

In [28]:
# response = llm.invoke("What is my name")
# response.content

## **RAG Implementation with PDFs**

### Step 1: Extracting text from PDF

In [29]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

pdf_path = "./Docs/"

loader = PyPDFDirectoryLoader(pdf_path)

docs = loader.load()
docs

[Document(metadata={'producer': 'Acrobat Distiller 22.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2026-07-15T03:44:50+05:30', 'author': 'Admin', 'moddate': '2026-07-15T03:44:50+05:30', 'title': 'Fabric administration documentation - Microsoft Fabric _ Microsoft Learn', 'source': 'Docs\\fabric_admin.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='/0/1/2/3/4/5/i255/1/7/8/4/9/4/10/11/3/1/11/4/12/9/i255/7/12/5/13/8/14/9/11/1/11/4/12/9\n/15/16/17/18/19/i255/17/21/22/23/24/i255/24/25/16/i255/26/17/21/18/27/28/i255/17/29/30/27/19/i255/31/16/24/24/27/19/32/31/33/i255/22/34/24/27/22/19/31/33/i255/17/19/29/i255/24/22/22/35/31/36\n/0/1/2/3/4/5/i255/4/9/i255/37/12/13/3/i255/12/3/38/1/9/4/39/1/11/4/12/9\n/40/41/42/43/44/42/45/43/46\n/47/25/17/24/i255/27/31/i255/17/29/30/27/19/27/31/24/18/17/24/27/22/19/i255/27/19/i255/26/17/21/18/27/28/48\n/49/50/43/51/i255/52/51/53/44/51/43/54\n/55/19/17/21/35/16/i255/26/17/21/18/27/28/i255/56/22/18/i255/57/22/23/18

In [30]:
# Extract the unique source file paths from each page's metadata
loaded_files = set(doc.metadata['source'] for doc in docs if 'source' in doc.metadata)

print(f"Total files successfully loaded: {len(loaded_files)}")
print("List of loaded files:")
for file in loaded_files:
    print(f" - {file}")


Total files successfully loaded: 1
List of loaded files:
 - Docs\fabric_admin.pdf


### Creating own metadata

In [31]:
# for i in docs:
#     i.metadata = {"source":"local_pdf",
#                   "author":"Microsoft"
#                   }

### Step 2: Splitting document to chunks 

In [32]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)

chunks = splitter.split_documents(docs)
len(chunks)


4

### Step 3: Creating Embeddings for the Chunks

In [33]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model = "text-embedding-3-small") 


### Step 4 : Using Chroma DB as vector store. Store embedding in the existing vector store

#### Langchain will take care of embedding of our each chunk and store it in vector store



In [34]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma(persist_directory="./Vector/",
                     embedding_function=embedding_model
)

vectorstore.add_documents(chunks)

# # In background, this happened - Langchain did for us

# # vectors = []
# # for doc in chunks:
# #     vector = embedding_model.embed_documents([doc.page_content])
# #     vectors.append(vector)

['78078e96-beed-493d-8685-0a195a776e1d',
 'c20468cb-cce0-4f55-a13a-04f048a6ec6e',
 '30e157ba-0e3b-43d1-9a96-112441fb032a',
 'af09da13-80c4-4f1f-8c2d-189df113b973']

### Similarity or Semantic Search

In [35]:
vectorstore.similarity_search("What is ms fabric admin?", k=3)

## User probes here

[Document(metadata={'page': 381, 'total_pages': 382, 'creator': 'Microsoft Learn', 'moddate': '2025-12-12T17:01:10+00:00', 'title': 'fabric onelake | Microsoft Learn', 'page_label': '382', 'producer': 'Microsoft Learn PDF 1.0.25309.01', 'creationdate': '2025-12-12T17:01:10+00:00', 'source': 'Docs\\fabric-onelake.pdf'}, page_content="OneLake security roles are propagated on the SQL analytics endpoint with the OLS_\nprefix.\nUser changes on the OLS_ roles are not supported, and can cause unexpected behaviors.\nMail enabled security groups and distribution lists are not supported.\nThe owner of the lakehouse must be a member of the admin, member, or contributor\nworkspace roles; otherwise, security isn't applied to the SQL analytics endpoint.\nThe owner of the lakehouse cannot be a service principal for security sync to work.\nBest practices to secure data in OneLake\nOneLake security access control model\nLast updated on 10/30/2025\nRelated content"),
 Document(metadata={'producer': 'Acr